<img src="https://github.com/ArchiColab/sitex/blob/main/images/banner.png?raw=1" width="1000"/>

# Walking & Cycling Isochrones for QGIS

**The question:** from here, can you walk or cycle to the school, the health station and the
market? This is the idea of the **15-minute city** (Moreno et al., 2021): everything people need
every day (school, health care, food, services) within a 15-minute walk or bike ride from home.
The objective of this notebook is to investigate which parts of the ward have their daily public
services within a 5, 10 or 15 minute walk or a 5 or 10 minute bike ride, and which don't.

> Moreno, C., Allam, Z., Chabaud, D., Gall, C., & Pratlong, F. (2021). Introducing the
> "15-Minute City": Sustainability, resilience and place identity in future post-pandemic
> cities. *Smart Cities*, 4(1), 93–111. https://doi.org/10.3390/smartcities4010006

Pick the amenities you want to study (kindergartens, commune health stations, markets ...),
pick a street network, and this notebook draws the areas you can reach on foot from them in
5, 10 and 15 minutes, and by bike in 5 and 10 minutes. The result is one GeoPackage, ready to drag into QGIS and use in a
layout.

The notebook is optimized for Google Colab. Because it runs in the cloud, it can be used
on machines with limited computing power and in areas with limited bandwidth.

<img src="https://github.com/ArchiColab/sitex/blob/main/images/Isochrone_overview.svg?raw=1" width="1000"/>

**Library:** [`sitex`](https://github.com/ArchiColab/sitex)

**Requires (run first):**
- `03-NA02-POIs_Gehl.ipynb`: the classified places
  (`data/overture/pois_classified_{slug}.geojson`).
- A street network: the OSM walk network saved during data acquisition (`osm/{slug}_walk.gpkg`),
  your own street layer drawn in QGIS, or nothing at all (the notebook can download one).
- `00-Data-Acquisition.ipynb`, Section 3b: the shared hex grid (`data/hex_grid_{slug}.gpkg`), for the
  export in Section 7.

**Output (`data/gdf/`):** `isochrones_{slug}.gpkg` for QGIS, and the cleaned places
`pois_amenities_{slug}.gpkg`, which the Simple Isochrone and Simple Service Area notebooks
read. Also `{slug}_hex_amenities.parquet` (the number of facilities within 15 minutes of each hexagon) and its
legend `{slug}_hex_amenities_legend.csv` (Section 7).

**What you edit:** the *Your choices* cell in Section 1, and, if needed, the *Name rules*
cell in Section 2. Everything else runs as is.

---
## Coordinate systems (CRS)

- **Analysis runs in the site's UTM zone** (metres; global UTM, not the local VN-2000 CRS). Buildings, streets and hexagons use it. Landsat and Sentinel-2 stay in UTM; Open Buildings tiles are brought into the site's UTM zone.; the terrain model, land cover and canopy height are saved in longitude/latitude (EPSG:4326) and are sampled or reprojected when a notebook needs metres. For initial analysis you can ignore VN-2000.
- **UTM is the default;** to combine with official VN-2000 data, reproject the vector data (see `00-Data-Acquisition`).
- **Sources of CRS:** the VN-2000 definitions are listed at [https://gis.vn/](https://gis.vn/kinh-tuyen-truc-ban-do-hanh-chinh-cap-tinh-thanh-viet-nam).

---
## 1. Setup & your choices

<img src="https://github.com/ArchiColab/sitex/blob/main/images/Isochrone_Setup.svg?raw=1" width="1000"/>

Run the first cell once. Then set your choices in the second cell:

| Setting | What it does |
|---|---|
| `AMENITIES` | What to draw isochrones for: whole **groups** ("Healthcare", "Education", ...) or single **types** ("Kindergarten", "Hospital", ...), mixed as you like. Each entry gets its own isochrones. Section 2 prints every group and type with its count. |
| `NETWORK_SOURCE` | `"files"`: build the network from the line layers in `NETWORK_FILES`. `"osm"`: download the OpenStreetMap walk network. |
| `NETWORK_FILES` | One or more line layers, paths inside the data folder. They are combined, so you can add a street you drew in QGIS (e.g. `osm/new_street.geojson`) to the existing network to test a proposal. |
| `ISOCHRONE_SHAPE` | `"concave"` (recommended) draws the outline around every street junction you can reach in time, as tight as possible while it stays one shape and contains every street you walked (Dolmajian, 2023). `"convex"` is the smoother convex hull, which also covers corners no street reaches. |
| `BIKE_SPEED_KMH`, `BIKE_TIME_BANDS` | The cycling speed (15 km/h) and the bike times in minutes, for Section 5. The bike uses the same street network as walking. |

In [1]:
#@title <font color=#1B7192> Configuration </font>  { display-mode: "form" }

# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

    from google.colab import drive
    drive.mount("/content/gdrive")

    DATA_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/SiteX_Outputs")

    if not (DATA_DIR / "overture").is_dir():
        raise FileNotFoundError(
            f"{DATA_DIR} not found (or incomplete) in your Google Drive. "
            "Before the workshop: download the workshop data zip and unzip it into "
            "My Drive/Colab_Outputs -- see the workshop setup instructions."
        )
else:
    DATA_DIR = Path("..") / "data"
    OUTPUT_DIR = Path("..") / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import geopandas as gpd
from IPython.display import display

from sitex.arch import poi_amenities as am
from sitex.core.config import CityConfig
from sitex.network import isochrones as iso

Running in: Local environment


In [2]:
#@title <font color=#1B7192> Your choices </font>  { display-mode: "form" }

# The place and short name of 00-Data-Acquisition (its PLACE and SLUG): same values, same files.
PLACE = "Phường Pleiku, Gia Lai, Vietnam"   # or an OSM ID, e.g. "R19260957"
SLUG = "pleiku"                              # "" = taken from the OSM name

city = CityConfig(
    place_name=PLACE,
    slug=SLUG,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
)

# Which amenities? Whole groups or single types from the table in Section 2, e.g.
#   ["Kindergarten", "Primary school", "Healthcare"]
AMENITIES = ["Healthcare", "Education", "Daily Needs & Grocery", "Civic & Government"]

# Which street network?  "files" = the line layers below, combined;  "osm" = download
NETWORK_SOURCE = "files"
NETWORK_FILES = [f"osm/{city.slug}_walk.gpkg"]  # add e.g. "osm/new_street.geojson" to test a new street

TIME_BANDS = [5, 10, 15]      # minutes
WALK_SPEED_KMH = 4.8
BIKE_SPEED_KMH = 15.0        # Schuhmacher et al. (2025), see Section 5
BIKE_TIME_BANDS = [5, 10]    # minutes by bike
ISOCHRONE_SHAPE = "concave"   # "concave" (recommended) or "convex"

# Facilities farther than this from any street are set aside, not snapped to a distant one
MAX_SNAP_M = 250

---
## 2. Clean & pick amenities

<img src="https://github.com/ArchiColab/sitex/blob/main/images/Isochrone_amenities.svg?raw=1" width="1000"/>

Overture's category is often wrong for the individual place: a veterinary clinic listed as
`hospital`, a motorbike dealer as `doctors_office`, an upper-secondary school as
`elementary_school`. The Vietnamese *name* is far more reliable ("Nha khoa", "Mầm non",
"THCS", "Trạm y tế"), so every place gets an `amenity_type` from its name. Places whose name
doesn't match a facility type, organisations named after a facility (a school's trade union,
a class reunion group), and private businesses named after another town (a branch listed in
the wrong place) are left out. Schools, public healthcare and offices are kept even when their
name mentions another town, because they are often named after a place.

**The name rules are in the next cell**, so you can read and change them. They work for
Vietnamese names anywhere in the country. For another city, the one list you must rewrite
is `OTHER_PLACES`: towns *outside your study area* whose name in a listing means the place
is really a branch somewhere else. If a type is missing, add a rule; if a rule catches the
wrong places, add a word to its "but not" list. Re-run the cell below to see the effect.

The table lists every amenity type with how many places were **kept** out of the total that
matched; ✓ marks the ones you chose. Below it, a sample of what was left out of your chosen
categories shows why the clean-up matters.

**Fixing mistakes in QGIS.** The first run writes `pois_amenity_review_{slug}.gpkg` to
`data/gdf/`, with every place and a `keep` column (1/0). Open it in QGIS, correct `keep` or
`amenity_type` where the automatic typing is wrong, save, and re-run this notebook: your edits
are applied by each place's id. Delete the file to start again from the automatic result.

In [3]:
#@title <font color=#1B7192> Name rules </font>  { display-mode: "form" }

# How to read a rule:
#   ("Amenity type", "Group", [name contains any of these], [but none of these])
#
# - Keywords are lowercase. Written WITHOUT accents they match names with or without
#   accents ("mam non" finds "Mầm Non" and "Mam Non"). Write a keyword WITH accents only
#   when dropping them would mix it up with another word ("chợ" market / "chó" dog / "cho").
# - Keywords match whole words: "bv" does not match inside "bvdk".
# - "^" means the name must START with the keyword ("^chợ"); "?" makes the character
#   before it optional ("pets?" = "pet" or "pets", "mini ?mart" = "minimart" or "mini mart").
# - The FIRST matching rule wins, so specific rules come before general ones:
#   "Veterinary" sits above "Hospital", so "Bệnh viện thú y" is not a hospital.

AMENITY_RULES = [
    # -- checked first, so they can't fall into a later rule --
    ("Veterinary / pet shop", "Pet Services", ["thu y", "thu cung", "thu canh", "pets?", "vet"], []),

    # -- Healthcare: the public network, by official level --
    ("Commune health station", "Healthcare", ["tram y te", "tram yte"], []),     # Trạm y tế
    ("Medical centre", "Healthcare", ["trung tam y te", "ttyt"], []),            # Trung tâm y tế
    ("Hospital", "Healthcare",                                                   # Bệnh viện
        ["benh vien", "bv", "bvdk", "hospital", "benh xa"],
        ["nha khoa"]),
    # Public polyclinics only: regional or ward level. A plain "phòng khám đa khoa" is private
    # (see Clinic). "phường" needs its accents: without them it reads as the name "Phương".
    ("Polyclinic", "Healthcare",                                                 # Phòng khám đa khoa khu vực / phường
        ["phong kham da khoa khu vuc", "pkdk khu vuc", "phong kham da khoa cap phuong",
         "phòng khám đa khoa phường", "pkđk phường"], []),

    # -- Private Healthcare: a separate group, so it stays out of the public "Healthcare" --
    ("Dental clinic", "Private Healthcare", ["nha khoa", "dental", "implant", "nieng rang"], []),
    ("Pharmacy", "Private Healthcare",
        ["nha thuoc", "quay thuoc", "hieu thuoc", "pharmacy", "pharmacity", "long chau"], []),
    ("Clinic", "Private Healthcare",                                             # Phòng khám
        ["phong kham", "pk", "clinic", "y hoc gia dinh", "phong tiem", "tiem chung", "tiem ngua",
         "xet nghiem", "medlatec", "medilab"],
        ["spa", "skin", "skinlab", "beauty", "tham my", "adn", "optic", "mat kinh"]),

    # -- Education: preschool & general education (Giáo dục mầm non và phổ thông) --
    # Multi-level first, so "Tiểu học và THCS" isn't typed as a primary school
    ("Multi-level school", "Education",                                          # Liên cấp
        ["lien cap", "tieu hoc va trung hoc", "tieu hoc va thcs", "tieu hoc thcs", "th va thcs",
         "th thcs", "thcs va thpt", "thcs thpt", "trung hoc co so va trung hoc pho thong",
         "pho thong nhieu cap"],
        ["luyen thi", "boi duong", "trung tam"]),
    ("Kindergarten", "Education",                                                # Mầm non
        ["mam non", "mau giao", "nha tre", "nhom tre", "lop mn", "preschool", "kindergarten",
         "montessori"], []),
    ("Primary school", "Education", ["tieu hoc", "truong th", "primary school"],  # Tiểu học
        ["tien tieu hoc"]),
    ("Lower secondary school", "Education",                                      # THCS
        ["thcs", "trung hoc co so", "secondary school", "middle school"], []),
    ("Upper secondary school", "Education",                                      # THPT
        ["thpt", "trung hoc pho thong", "high school", "pho thong dan toc noi tru"], []),

    # -- Education: continuing education (Giáo dục thường xuyên) --
    # Only names with GDTX: "GDNN" alone also names beauty and driving schools
    ("Continuing education centre", "Education", ["giao duc thuong xuyen", "gdtx"], []),

    # -- Education: higher education (Giáo dục đại học) --
    # "học viện" needs its accents ("học viên" = student) and is also used by hair, nail,
    # makeup and yoga "academies", hence the long "but not" list
    ("University", "Education",                                                  # Đại học, học viện
        ["dai hoc", "học viện", "university"],
        ["luyen thi", "on thi", "gia su", "tuyen sinh", "toc", "nail", "mi", "makeup", "make up",
         "tham my", "spa", "yoga", "beauty", "barber", "kinh doanh", "anh ngu", "tieng anh"]),

    # -- Education: vocational (Giáo dục nghề nghiệp), a separate system from đại học --
    ("Vocational college", "Education", ["cao dang", "trung cap"], ["tuyen sinh"]),  # Cao đẳng, trung cấp

    # -- Education: private courses --
    ("Language / tutoring centre", "Education",
        ["anh ngu", "anh van", "tieng anh", "ngoai ngu", "tieng trung", "tieng nhat", "tieng han",
         "english", "luyen thi", "gia su", "boi duong"],
        ["du hoc", "viec lam"]),

    # -- Daily Needs & Grocery --
    ("Market", "Daily Needs & Grocery", ["^chợ", "cho dau moi", "market"], ["gourmet"]),
    ("Supermarket", "Daily Needs & Grocery",
        ["sieu thi", "coopmart", "co opmart", "winmart", "vinmart", "bach hoa xanh", "lotte mart",
         "big c", "emart", "mm mega"],
        ["dien may", "di dong", "topzone", "laptop", "thiet bi", "noi that", "nem", "cua", "o ?to",
         "bep", "thu cung", "nong nghiep", "kim cuong", "lotteria", "gia ke"]),
    ("Convenience / grocery store", "Daily Needs & Grocery",
        ["tap hoa", "bach hoa", "cua hang tien loi", "cua hang tien ich", "circle k", "family ?mart",
         "ministop", "gs25", "7 eleven", "mini ?mart", "combini"], []),

    # -- Civic & Government --
    ("People's Committee", "Civic & Government", ["uy ban nhan dan", "ubnd"], []),
    ("Police", "Civic & Government", ["cong an", "police"], ["benh xa"]),
    ("Government office", "Civic & Government",
        ["^sở", "^so giao duc", "^so y te", "^so noi vu", "^so tai chinh", "^so xay dung",
         "^so khoa hoc", "^so khcn", "^so nong nghiep", "^so lao dong", "^so tu phap",
         "^so cong thuong", "^so ke hoach", "^so van hoa", "^so tai nguyen", "^so giao thong",
         "chi cuc", "bao hiem xa hoi", "bhxh", "kho bac"], []),

    # -- Community & Culture --
    ("Pagoda / temple", "Community & Religion",
        ["chùa", "tinh xa", "pagoda", "temple"],
        ["tea", "coffee", "cafe", "tour", "travel"]),
    ("Church", "Community & Religion",
        ["nha tho", "giao xu", "hoi thanh", "tin lanh", "httl", "church"], []),
    ("Library", "Culture & Arts", ["thu vien", "library"], []),
    ("Museum", "Culture & Arts", ["bao tang", "museum"], []),
    ("Park / square", "Recreation & Sport",
        ["cong vien", "quang truong", "park"],
        ["can ho", "chung cu", "residence"]),
    ("Sports ground", "Recreation & Sport", ["san bong", "san van dong", "nha thi dau", "stadium"], []),
]

# Organisations named after a facility (a school's trade union or youth union, a class
# reunion group, a party cell, a school "confessions" page): left out, they are not the facility
ORGANISATION_KEYWORDS = [
    "cong doan", "cdcs", "chi doan", "chi bo", "doan co so", "dcs", "lien doi", "ld truong",
    "class", "cuu hoc sinh", "hoi dong doi", "confessions?", "tuoi tre", "doan truong",
    "hoi sinh vien", "lien chi hoi", "sinh vien 5 tot", "doan thanh nien",
]

# Towns OUTSIDE your study area: a listing whose name mentions one is a branch somewhere
# else that the map data placed here by mistake. Rewrite this list for another city.
OTHER_PLACES = [
    "chu se", "chu pah", "chu puh", "chu ?prong", "kon ?tum", "komtum", "quy nhon", "ayun ?pa",
    "an khe", "duc co", "phu thien", "dak doa", "kbang", "mang den", "ia ?grai", "phan thiet",
    "ha noi", "quang ngai",
]

# Official facilities are often NAMED AFTER a place ("THPT Trần Quốc Tuấn, Phú Thiện"), so
# for these types a town name is not a sign of a wrong location and they are kept. One that
# really lies elsewhere is still set aside in Section 3, by its distance to the streets.
OFFICIAL_TYPES = [
    "Kindergarten", "Primary school", "Lower secondary school", "Upper secondary school",
    "Multi-level school", "Continuing education centre", "University", "Vocational college",
    "Commune health station", "Polyclinic", "Medical centre", "Hospital",
    "People's Committee", "Police", "Government office",
]

print(f"{len(AMENITY_RULES)} amenity rules, {len(ORGANISATION_KEYWORDS)} organisation keywords, "
      f"{len(OTHER_PLACES)} other towns, {len(OFFICIAL_TYPES)} official types")

29 amenity rules, 18 organisation keywords, 18 other towns, 15 official types


In [4]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 2. Clean & pick amenities ###", flush=True)

pois = gpd.read_file(city.data_dir / "overture" / f"pois_classified_{city.slug}.geojson")
pois = pois[pois.geometry.type == "Point"].reset_index(drop=True)
pois = am.classify_amenities(pois, AMENITY_RULES, ORGANISATION_KEYWORDS, OTHER_PLACES, OFFICIAL_TYPES)

review_path = city.data_dir / "gdf" / f"pois_amenity_review_{city.slug}.gpkg"
if review_path.exists():
    pois, n_changed = am.apply_review_file(pois, review_path, AMENITY_RULES)
    print(f"Applied your QGIS review ({review_path.name}): {n_changed} edits")
else:
    am.write_review_file(pois, review_path, city.local_epsg)
    print(f"Wrote {review_path} -- open it in QGIS to correct the typing")

table = am.amenity_summary(pois)
chosen = table["amenity_type"].isin(AMENITIES) | table["amenity_group"].isin(AMENITIES)
table["chosen"] = chosen.map({True: "✓", False: ""})

facilities, unknown = am.select_amenities(pois, AMENITIES)
if unknown:
    print(f"Not found, check the spelling: {unknown}")
print(f"{len(facilities)} facilities chosen from {len(pois):,} places:")
print(facilities["amenity"].value_counts().reindex(AMENITIES).dropna().astype(int).to_string())

print(f"--- 2. done in {time.time() - _t0:.2f}s ---\n")
display(table)

chosen_groups = table.loc[table["chosen"] == "✓", "amenity_group"].unique()
left_out = pois[pois["functional_group"].isin(chosen_groups) & ~pois["keep"]]
print(f"Left out of your chosen categories: {len(left_out)} places, for example:")
display(left_out[["name", "overture_category", "note"]].sample(min(12, len(left_out)), random_state=0))

### 2. Clean & pick amenities ###
Applied your QGIS review (pois_amenity_review_pleiku.gpkg): 0 edits
135 facilities chosen from 3,281 places:
amenity
Healthcare               20
Education                64
Daily Needs & Grocery    29
Civic & Government       22
--- 2. done in 1.65s ---



,amenity_group,amenity_type,kept,total,chosen
0,Civic & Government,Government office,12,12,✓
1,Civic & Government,People's Committee,7,7,✓
2,Civic & Government,Police,3,3,✓
3,Community & Religion,Church,11,11,
4,Community & Religion,Pagoda / temple,9,9,
5,Culture & Arts,Museum,2,2,
6,Culture & Arts,Library,1,1,
7,Daily Needs & Grocery,Convenience / grocery store,16,16,✓
8,Daily Needs & Grocery,Supermarket,11,12,✓
9,Daily Needs & Grocery,Market,2,3,✓


Left out of your chosen categories: 173 places, for example:


,name,overture_category,note
1159,"Thông Cống Nghẹt ,Hút Hầm Cầu Pleiku giá rẻ",community_and_government,no rule matched
2794,Vật tư Y tế Gia Lai - Cty CP XNK Y tế Gia Lai,health_care,no rule matched
1361,Truong Tiep Hoc Tran Dai Nghia,elementary_school,no rule matched
1189,Cơ Sở Chế Biến Thực Phẩm Thiên Phú,health_food_store,no rule matched
2457,Họa cụ Palette_Gia lai,grocery_store,no rule matched
1898,Trung TÂM ToÁN Smart Kids,education,no rule matched
129,Việc Làm - Ngoại Ngữ - Du Học Phương Đông,education,no rule matched
3095,CƠ Quan ThƯỜNg TrÚ Ttxvn TẠI TỈNh Gia Lai,government_office,no rule matched
2018,Học Thanh Nhạc Cho Người Mới Bắt Đầu,elementary_school,no rule matched
1734,Nature School,education,no rule matched


---
## 3. Street network

<img src="https://github.com/ArchiColab/sitex/blob/main/images/Isochrone_network.svg?raw=1" width="1000"/>

With `"files"`, the line layers are combined, snapped to a 1 m grid and split wherever two
lines cross, so a street you drew in QGIS joins the existing network where it touches it
(turn on snapping while drawing). Only the largest connected part of the network is kept: a
street that doesn't touch the rest can't be reached from anywhere. Every street gets the same
walking speed.

Each facility is then attached to its nearest street junction. Facilities more than
`MAX_SNAP_M` from any street lie outside the network's area and are set aside.

In [5]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 3. Street network ###", flush=True)

if NETWORK_SOURCE == "osm":
    print(f"Downloading the OSM walk network for {city.place_name} ...")
    G = iso.network_from_osm(city.place_name, city.local_epsg, WALK_SPEED_KMH)
else:
    G = iso.network_from_files([city.data_dir / f for f in NETWORK_FILES], city.local_epsg, WALK_SPEED_KMH)
print(f"Network: {G.number_of_nodes():,} junctions, {G.number_of_edges() // 2:,} street segments")

facilities_on, facilities_off = iso.snap_facilities(G, facilities, MAX_SNAP_M)
print(f"{len(facilities_on)} facilities on the network, {len(facilities_off)} set aside (> {MAX_SNAP_M} m from any street)")
if len(facilities_off):
    print(facilities_off[["name", "amenity", "snap_dist_m"]].to_string(index=False))

print(f"--- 3. done in {time.time() - _t0:.2f}s ---\n")

### 3. Street network ###
Network: 2,687 junctions, 3,364 street segments
134 facilities on the network, 1 set aside (> 250 m from any street)
               name   amenity  snap_dist_m
Trường Mầm Non 19/8 Education        268.1
--- 3. done in 1.11s ---



---
## 4. Isochrones

<img src="https://github.com/ArchiColab/sitex/blob/main/images/Isochrone_isochrones.svg?raw=1" width="1000"/>

From every facility, the notebook walks outward along the network up to the longest time
band. The facilities of each entry in `AMENITIES` are then merged: the 10-minute area of
"Healthcare" is everywhere within 10 minutes of the *nearest* public health facility, whatever
its type. The table gives each area in hectares; the map lets you switch between the entries.

Each area is drawn as a concave hull around every street junction you can reach in time
(Dolmajian, 2023): as tight as possible while it stays one shape and contains every street
you walked. Where that is not possible it becomes the convex hull. The edge is approximate,
not an exact line. The walk doesn't stop at the last junction reached: where the time runs out halfway along a
street, that point counts too, and so does the facility itself (Dolmajian, 2023).

**Why this shape, and why no buffer.** We tested the shapes on Pleiku's 21,389 building
footprints, for healthcare facilities and for kindergartens. A building counts as truly
reachable when the walk along the streets, plus the short walk from the street to the
building, fits in the time. The concave hull matched best: it includes about 96% of the truly
reachable buildings, and 93–95% of the buildings inside it are truly reachable. A buffer
around the walked streets did worse at every width from 15 m to 100 m (best at 50 m, which
still takes in more buildings that are not reachable), so no buffer is used. The bands are
also merged so that they always nest: the 10-minute area always contains the 5-minute one.

> Dolmajian, A. (2023). *Creating isochrones: what is the optimal way?* Medium.
> https://medium.com/@arthur.dolmajian/creating-isochrones-what-is-the-optimal-way-dfc77a2ca13a

In [6]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 4. Isochrones ###", flush=True)

facility_iso = iso.facility_isochrones(G, facilities_on, TIME_BANDS, shape=ISOCHRONE_SHAPE)
isochrones = iso.dissolve_isochrones(facility_iso)
bands = iso.isochrone_bands(isochrones)

print(isochrones.pivot(index="amenity", columns="time_min", values="area_ha").add_suffix(" min (ha)").to_string())

print(f"--- 4. done in {time.time() - _t0:.2f}s ---\n")
iso.preview_map(bands, facilities_on)

### 4. Isochrones ###
time_min               5 min (ha)  10 min (ha)  15 min (ha)
amenity                                                    
Civic & Government          204.5        504.0        746.7
Daily Needs & Grocery       346.3        763.7       1165.5
Education                   461.6        796.9       1123.7
Healthcare                  268.2        808.8       1247.9
--- 4. done in 144.44s ---



---
## 5. By bike

The 15-minute city counts both walking and cycling (Moreno et al., 2021). This section rides
the **same street network** by bike, at `BIKE_SPEED_KMH` = 15 km/h: the low end of the median
speeds Schuhmacher et al. (2025) measured for cyclists in Hamburg, 15.0–21.7 km/h depending on
the street. The low end fits Pleiku, where cyclists share the streets with motorbikes and the
plateau has slopes.

**Why 5 and 10 minutes, not 15.** At 15 km/h a 15-minute ride covers about 3.75 km, which
reaches almost the whole ward, so the map would show little. A **5-minute ride (about 1.25 km)
reaches about as far as a 15-minute walk (1.2 km)**: compare the two on the map and in the table.

Alleys, footpaths and steps in the walk network count as rideable, so in places the bike areas
are a little too generous.

The table compares, for each entry in `AMENITIES`, the area of the longest walk with the areas by
bike; the map shows the bike bands.

> Schuhmacher, L., Wilkes, G., Kagerbauer, M., & Vortisch, P. (2025). Cycling speeds in
> urban traffic. *Findings*. https://doi.org/10.32866/001c.141204

In [7]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 5. By bike ###", flush=True)

import pandas as pd

G_bike = iso.with_speed(G, BIKE_SPEED_KMH)  # the same streets, ridden at BIKE_SPEED_KMH
bike_facility_iso = iso.facility_isochrones(G_bike, facilities_on, BIKE_TIME_BANDS, shape=ISOCHRONE_SHAPE)
bike_isochrones = iso.dissolve_isochrones(bike_facility_iso)
bike_bands = iso.isochrone_bands(bike_isochrones)

def _area(df, t):
    return df[df["time_min"] == t].set_index("amenity")["area_ha"]

compare = pd.DataFrame(
    {f"walk {max(TIME_BANDS)} min (ha)": _area(isochrones, max(TIME_BANDS))}
    | {f"bike {t} min (ha)": _area(bike_isochrones, t) for t in BIKE_TIME_BANDS}
)
print(compare.to_string())

print(f"--- 5. done in {time.time() - _t0:.2f}s ---\n")
iso.preview_map(bike_bands, facilities_on, mode="Bike")

### 5. By bike ###
                       walk 15 min (ha)  bike 5 min (ha)  bike 10 min (ha)
amenity                                                                   
Civic & Government                746.7            771.7            1172.4
Daily Needs & Grocery            1165.5           1256.3            2329.4
Education                        1123.7           1185.1            2168.8
Healthcare                       1247.9           1302.0            2036.5
--- 5. done in 308.91s ---



---
## 6. Export to QGIS

<img src="https://github.com/ArchiColab/sitex/blob/main/images/Isochrone_export.svg?raw=1" width="1000"/>

Everything goes into one GeoPackage in `data/gdf/`, in the local metric coordinate system
(`city.local_epsg`), so lengths and areas are in metres in QGIS:

| Layer | Contents |
|---|---|
| `isochrone_bands` | Non-overlapping rings (`band` = "0-5 min", "5-10 min", ...) per `amenity`. **The layer for your layout.** |
| `isochrones` | The full areas per amenity and time, nested (the 15-minute area contains the 5-minute one). |
| `bike_isochrone_bands`, `bike_isochrones` | The same two layers by bike (`BIKE_TIME_BANDS`). Put them next to the walking layers, e.g. the 15-minute walk against the 5-minute ride. |
| `service_areas` | Each facility's own largest-band area, kept separate; they overlap. |
| `facilities` | The facilities used, with `amenity` (your entry), `amenity_type` and `snap_dist_m`. |
| `network` | The street network the walks ran on. |

The cleaned places themselves (every place with its `amenity_type`, `amenity_group` and
`keep`) are also saved, as `data/gdf/pois_amenities_{slug}.gpkg`. They are rewritten on every
run and are the input for the Simple Isochrone and Simple Service Area notebooks.

**In QGIS:** drag the file into the Layers panel. Style `isochrone_bands` as *Categorized* on
`band`. To show one amenity per map, give each map its own copy of the layer with a
filter such as `"amenity" = 'Healthcare'`. For one page per amenity, use an Atlas with
`isochrones` as coverage layer and `amenity` as page name.

In [8]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 6. Export to QGIS ###", flush=True)

out_path = iso.export_geopackage(
    city.data_dir / "gdf" / f"isochrones_{city.slug}.gpkg",
    {
        "isochrone_bands": bands,
        "isochrones": isochrones,
        "bike_isochrone_bands": bike_bands,
        "bike_isochrones": bike_isochrones,
        "service_areas": iso.service_areas(facility_iso),
        "facilities": facilities_on.drop(columns=["node"]),
        "network": iso.network_edges(G),
    },
)
print(f"Saved {out_path}")

pois_path = am.export_amenities(pois, city.data_dir / "gdf" / f"pois_amenities_{city.slug}.gpkg", city.local_epsg)
print(f"Saved {pois_path} (the cleaned places, for later notebooks)")

print(f"--- 6. done in {time.time() - _t0:.2f}s ---\n")

### 6. Export to QGIS ###


Saved ..\data\gdf\isochrones_pleiku.gpkg
Saved ..\data\gdf\pois_amenities_pleiku.gpkg (the cleaned places, for later notebooks)
--- 6. done in 0.44s ---



---
## 7. Export for interpretation: facilities within 15 minutes of each hexagon

Every analysis notebook ends with a table like this one: its results per hexagon of the shared
hex grid (`hex_grid_{slug}.gpkg`, built in `00-Data-Acquisition.ipynb` Section 3b), keyed by
`h3_id`. The tables from different notebooks can then be joined and read side by side. The
numbers are **per hexagon, not per resident**.

For each hexagon, the notebook attaches the hexagon's centre to the nearest street junction,
walks 15 minutes along the network from there, and counts the facilities it reaches. The count
covers the facilities of your `AMENITIES` choice, so a different choice gives a different
count. `snap_dist_m` is the distance from the hexagon centre to its junction: a large value means
the hexagon is far from any street.

Only a count is exported, not the types: the types need a survey on site. The count says how
many places are in reach, not how good they are.

The file `{slug}_hex_amenities.parquet` and its legend `{slug}_hex_amenities_legend.csv` are
written to `data/gdf`.


In [9]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 7. Export for interpretation: facilities within 15 minutes of each hexagon ###", flush=True)

import h3
import pandas as pd

hex_grid_path = city.data_dir / f"hex_grid_{city.slug}.gpkg"
if not hex_grid_path.exists():
    raise FileNotFoundError(f"{hex_grid_path} not found -- run 00-Data-Acquisition.ipynb Section 3b first.")
hexes = gpd.read_file(hex_grid_path, layer="hex")

hex_amenities = iso.count_facilities_by_hex(G, hexes, facilities_on, minutes=15)
hex_amenities = hex_amenities.rename(columns={"n_amenities": "n_amenities_15min"})
hex_amenities_path = city.data_dir / "gdf" / f"{city.slug}_hex_amenities.parquet"
hex_amenities.to_parquet(hex_amenities_path, index=False)

# --- legend: one dictionary writes the table below and the legend file ---
H3_RES = h3.get_resolution(hexes["h3_id"].iloc[0])
LEGEND_COLUMNS = ["table", "column", "unit", "definition", "source", "limits"]
LEGEND = [
    ("hex_amenities", "h3_id", "-", f"Key of the hexagon (H3 cell, resolution {H3_RES})", f"hex_grid_{city.slug}.gpkg", "-"),
    ("hex_amenities", "n_amenities_15min", "count",
     f"Places of {', '.join(AMENITIES)} within 15 min of walking at {WALK_SPEED_KMH} km/h from the hexagon centre",
     "Overture Maps places, cleaned by the name rules (Section 2); street network of this run (Section 3)",
     f"Counts places, not their size or quality; every type counts once. Places more than {MAX_SNAP_M} m from a street "
     "are not counted. Follows the AMENITIES choice and any street added to the network"),
    ("hex_amenities", "snap_dist_m", "m", "Distance from the hexagon centre to the street junction the walk starts from",
     "street network of this run", "The walk from the centre to the junction is not counted. A large value: the hexagon is far from any street"),
]
legend = pd.DataFrame(LEGEND, columns=LEGEND_COLUMNS)
legend_path = city.data_dir / "gdf" / f"{city.slug}_hex_amenities_legend.csv"
legend.to_csv(legend_path, index=False, encoding="utf-8")

print(f"Hexagons: {len(hex_amenities)}, of which {int((hex_amenities['n_amenities_15min'] == 0).sum())} with no facility within 15 min")
print(f"Saved -> {hex_amenities_path}")
print(f"Saved -> {legend_path}")
display(hex_amenities.drop(columns="h3_id").describe().round(1))
display(legend.drop(columns="table"))

print(f"--- 7. done in {time.time() - _t0:.2f}s ---\n")


### 7. Export for interpretation: facilities within 15 minutes of each hexagon ###
Hexagons: 217, of which 67 with no facility within 15 min
Saved -> ..\data\gdf\pleiku_hex_amenities.parquet
Saved -> ..\data\gdf\pleiku_hex_amenities_legend.csv


,n_amenities_15min,snap_dist_m
count,217.0,217.0
mean,10.2,90.4
std,17.2,85.3
min,0.0,6.4
25%,0.0,34.5
50%,2.0,62.1
75%,10.0,109.9
max,72.0,451.9


,column,unit,definition,source,limits
0,h3_id,-,"Key of the hexagon (H3 cell, resolution 9)",hex_grid_pleiku.gpkg,-
1,n_amenities_15min,count,"Places of Healthcare, Education, Daily Needs &...","Overture Maps places, cleaned by the name rule...","Counts places, not their size or quality; ever..."
2,snap_dist_m,m,Distance from the hexagon centre to the street...,street network of this run,The walk from the centre to the junction is no...


--- 7. done in 0.32s ---



---
## 8. Field check: a 15-minute neighbourhood?

The 15-minute city is a claim about daily life: when the school, the health station and the
market are close, people walk or cycle to them. The maps in Sections 4 and 5 only show where
that is *possible* by distance. The field check asks whether people actually live that way, and
if not, why not.

Before the field trip, pick 1–2 places for each row:

| Pattern on the map | What the model predicts | What to check on the ground |
|---|---|---|
| Inside the 10-minute walk area of healthcare, education *and* markets | A walkable neighbourhood for daily needs | Walk one route with a stopwatch. Did it take 10 minutes? |
| Outside the 15-minute walk area, but inside the 5-minute bike area | Daily needs are close by bike, not on foot | Do people cycle here? What stops them: motorbike traffic, slopes, heat, nowhere to leave a bike? |
| Inside one group's area, outside another's | One daily need forces a longer trip | Which need is it, and how do people solve it? |
| Outside every area, even by bike | Poorly served | The edge of the study area, or new development (see the urban-change map)? |

**What to survey.**

- **One route** from a home street to a facility: the time on a stopwatch and the time the map
  gives; every obstacle: motorbikes parked on the footpath, missing footpath, a road you have to
  cross, a slope, no shade.
- **At the facility**, at opening or closing time, for 15 minutes: how many people arrive on
  foot, by bicycle, by motorbike and by car. Ask one parent or patient how far they came, and why
  they came that way.
- **On one street of the route**, for 5 minutes: count pedestrians, cyclists and motorbikes. Note
  where cyclists ride (at the kerb, on the footpath, among the motorbikes), and whether there is
  anywhere to leave a bike.
- **What people do on the way:** stop at a shop, buy breakfast, talk to a neighbour. Trips that
  combine several daily needs are at the heart of the 15-minute idea.

*Example of a record:* "Street A to kindergarten B: map 8 minutes on foot (5–10 min band) and
under 5 minutes by bike; stopwatch 12 minutes on foot. Footpath blocked by parked motorbikes on
three stretches, so we walked on the road; one crossing of a four-lane road without lights. At
the gate, 16:30–16:45: 30 children collected, 26 by motorbike, 3 on foot, 1 by bicycle. On the
street outside, 5 minutes: 64 motorbikes, 2 bicycles, 9 pedestrians. A mother on foot said she
walks because she buys vegetables at the market on the way."

**Discuss:** the isochrones show what *could* be reached on foot or by bike, not how people
travel. Most trips here are made by motorbike, even short ones. Is the ward a 15-minute city by
distance but not in practice? What would have to change (shade, safe crossings, slower traffic,
somewhere to leave a bike) for a 10-minute walk or a 5-minute ride to become the normal way?

---
## Notes

- **The name rules are for Vietnamese names.** For another country, rewrite the keywords in
  the *Name rules* cell (Section 2) in that language.
- **The rules can't catch everything.** A place with a misleading name needs the QGIS review.
  Unnamed places are always left out.
- **Walking speed is a constant 4.8 km/h**, the speed Transport for London uses for walk times
  (Transport for London, 2015, *Assessing transport connectivity in London*, Table 2.1,
  https://content.tfl.gov.uk/connectivity-assessment-guide.pdf).
  Slopes, road crossings, missing footpaths and heat
  are not taken into account, so real walking times can be longer.
- **Cycling speed is a constant 15 km/h** on the same network (Schuhmacher et al., 2025, see
  Section 5). Slopes and traffic are not taken into account.
- **The edge of an isochrone is approximate.** A hull also covers some corners between
  streets. `"convex"` covers more of them than the default `"concave"`.

---
## When the model is wrong

Where your stopwatch and the map disagree, write down why. Common reasons:

- **The bike rides at a constant 15 km/h** on every street of the walk network, including
  alleys, steps and slopes. The map knows nothing about the motorbike traffic that decides
  whether cycling feels safe.
- **Every street is walked at 4.8 km/h.** Blocked footpaths, crossings, slopes and heat are not
  in the model.
- **Streets that aren't really walkable** (a highway without crossings) or **paths that are
  missing from the map** (a shortcut through a market) change the walk.
- **A facility just outside the study area** may be the one people really use.
- **Nearest isn't chosen.** People pick a school or clinic for its quality, price or opening
  hours, not only its distance.

Write one sentence per disagreement: *"The map says …, I found …, because …"*.

## Where to go next

| Next step | Open |
|---|---|
| One kind of facility, or test a new facility or a new street link | [`03-NA04-Simple_Isochrone.ipynb`](03-NA04-Simple_Isochrone.ipynb) |
| Which facility serves which part of the ward, and how many buildings | [`03-NA05-Simple_Service_Area.ipynb`](03-NA05-Simple_Service_Area.ipynb) |
| Walking time to green space | [`03-NA06-Green_Accessibility.ipynb`](03-NA06-Green_Accessibility.ipynb) |

**Into the design phase:** areas outside the bands are where a new facility or a missing street
link would help most. Draw the proposal in QGIS and test it in [`03-NA04-Simple_Isochrone.ipynb`](03-NA04-Simple_Isochrone.ipynb).